mini vector search, HNSW, IVF, PQ

https://arxiv.org/pdf/1301.3781

In [1]:
import numpy as np
import pandas as pd

In [4]:
import sqlite3
from sentence_transformers import SentenceTransformer

db = sqlite3.connect("vectors.db")
db.execute("CREATE TABLE IF NOT EXISTS docs(id INTEGER PRIMARY KEY, document TEXT UNIQUE, embedding BLOB)")
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
db.execute("DROP TABLE docs")

In [5]:
docs = [
    "cats are pets", "stocks fell today", "dogs love walks"
       # animals
    "lions hunt at dusk",
    "elephants never forget faces",
    "dolphins are very intelligent",
    "owls hunt at night",
    "cats chase small mice",
    "bees make sweet honey",
    # in between
    "counting sheep before sleep",
    "ants follow shortest paths",
    "honeycomb cells are hexagons",
    "sunflower seeds follow Fibonacci",
    "population growth of rabbits",
    # mathematics
    "prime numbers never end",
    "solving quadratic equations",
    "derivatives measure rate change",
    "matrix multiplication rules",
    "probability of rolling dice",
    "the Pythagorean theorem",
    "integrals compute area under curves",
]

embs = model.encode(docs, normalize_embeddings=True)
print(embs.shape)
db.executemany(
    "INSERT INTO docs(embedding, document) VALUES (?, ?)",
    [(e.tobytes(), d) for d, e in zip(docs, embs)],
)
db.commit()

(20, 384)


In [6]:
rows = db.execute("SELECT id, document, embedding FROM docs").fetchall()
rows[0]

(1,
 'cats are pets',
 b'd\x9e\x80=i\x8b\x1d<\xd5}W=\x902M=\xe9\xd7\xf4\xbd\x92|\x9a\xbc\xcb\xae\x8f<\x88\'\x8b\xbdo1\xb3<\xce\xd8\xd8\xbb\\\xef\xbb\xbc]3\xd0\xbc\x19\xab\xe7\xbb\xd7\x10\x17=\x81\xc9\xab\xbc\x92\x1bo\xbc\n\xaf\xaf\xbd\xd6\xba\x9a<M\xf9\xad\xbc\x82\xbch=\xb4\x9e\xa9\xbdc\xff\x94\xba;Y\xe8\xbc\x02LP\xbc\x06\xd9\x86\xbd\xecA\xa1=\xce\xee\x88\xbd\xc75I\xbdu\xed\xb2\xbcg\xb0\x8c=\xa9\xa98\xbd4\x98D=\x89\x05\x07\xbd(\xa9l=r\xfe8\xbd\xfcZ\xa2\xbc\xc1\x80\xe7<\x9f \xae<\xf9\x86\x8f=\xc2\x95\x8c=\x91\xa9p\xbd\x14\xf6\xb4\xbdJ}\xef;\xb2\x90j;f\x19V\xbdw,\x9b<\xf2b\xda\xbc\xc5\x1f\xb1\xbdBs\xb2<\xed8x\xbd\xadKa\xbd\xc2\xf2\x94=5\x1d\x82\xbd\x15\x9e\xa9;"\xf3\x05\xbcUms\xbd\xb8[\xd2<i&r\xbbL\xb1\xb1\xbc\x06\x08>\xbd\xeat\xfc<P\xc5\xb9<\xadk\xe7<:\xba\xae=\'\xcc\xc4<\t\xd6~\xbc\xc7\xa9\x1c\xbd\xdd\xa5F=\x97\xa1\xd2;\xd4"$\xbb 4\xc4<L\xd4]<\xc8\x98\xd2\xba\x91\xf3\x80\xbcQ:);\xady\xac<\x145_=\xe8\xefs<\xfb\x95-=E\xa2\x91\xbc9\x9d\xb1\xbc"M\xfc\xbc\xb4\x03s<o\x14&=\xe2vU=\xfe\x94\x07

In [7]:
M = np.vstack([np.frombuffer(r[2], dtype=np.float32) for r in rows])
M.shape

(20, 384)

In [8]:
# search
q = model.encode("animals", normalize_embeddings=True).astype(np.float32)

scores = M @ q # same brute force as sqlite-vec
scores

array([ 0.57389677,  0.12402201,  0.44344047,  0.3413345 ,  0.4115907 ,
        0.38987795,  0.42667672,  0.19147144,  0.35893333,  0.30212477,
        0.067072  ,  0.03711689,  0.30982083, -0.01484514,  0.10010544,
        0.02525808,  0.04732207,  0.09224994,  0.07068685,  0.08495688],
      dtype=float32)

In [9]:
nprows = np.vstack([r[1] for r in rows])
ixs = np.argsort(-scores)
frame = np.column_stack([nprows[ixs].squeeze(1), scores[ixs]])
df = pd.DataFrame(frame)
df

,0,1
0,cats are pets,0.57389677
1,dogs love walkslions hunt at dusk,0.44344047
2,cats chase small mice,0.42667672
3,dolphins are very intelligent,0.4115907
4,owls hunt at night,0.38987795
5,counting sheep before sleep,0.35893333
6,elephants never forget faces,0.3413345
7,population growth of rabbits,0.30982083
8,ants follow shortest paths,0.30212477
9,bees make sweet honey,0.19147144


In [10]:
frame.shape

(20, 2)

approximate nearest neighbor search (ANN) O(n) is too slow for millions of entries
- HNSW - generated graph that links to nearest neighbor. greedy hopping O(logn)
- IVF - k-means cluster them, only cosine sim on that grouped cluster/bucket or nearest ones
- PQ - quantization for mem efficiency
- LSH

Hierarchical Navigable Small World (Malkov and Yashunin in 2016)
- using 3 layers here < 1000 entries, but scale up in practice

THIS IS CLAUDE. ok i tried implementing it. got lost. so maybe ill try again another time

In [ ]:
"""HNSW (Malkov & Yashunin) — layer 0 as a fixed-width matrix, upper layers as dicts.

Off-the-shelf pieces used:
  numpy.random.Generator.geometric  – random level (geometric distribution)
  heapq                              – candidate / result priority queues
  numpy (matmul, argsort, argpartition, pad) – distances, ranking, growth
"""
import heapq
import numpy as np


class HNSW:
    def __init__(self, dim, M=16, ef_construction=200, capacity=1024, seed=None):
        self.M, self.M0, self.efc = M, 2 * M, ef_construction
        self.rng = np.random.default_rng(seed)
        self.n, self.entry, self.max_level = 0, -1, -1
        self.vecs = np.zeros((capacity, dim), dtype=np.float32)       # one row per node
        self.layer0 = np.full((capacity, self.M0 + 1), -1, dtype=np.int32)  # [count | nbrs...]
        self.layer0[:, 0] = 0
        self.upper = []                                               # upper[l-1] = {node: [nbrs]}

    # ---------- storage ----------
    def neighbors(self, node, layer):
        if layer == 0:
            return self.layer0[node, 1:1 + self.layer0[node, 0]]
        return self.upper[layer - 1].get(node, [])

    def set_neighbors(self, node, layer, nbrs):
        if layer == 0:
            row = self.layer0[node]
            row[1:] = -1
            row[0] = len(nbrs)
            row[1:1 + len(nbrs)] = nbrs
        else:
            self.upper[layer - 1][node] = list(nbrs)

    def _grow(self):                                                  # double capacity
        extra = len(self.vecs)
        self.vecs = np.pad(self.vecs, ((0, extra), (0, 0)))
        self.layer0 = np.pad(self.layer0, ((0, extra), (0, 0)), constant_values=-1)
        self.layer0[-extra:, 0] = 0

    def _dist(self, q, ids):                                          # cosine distance (normalized)
        return 1 - self.vecs[ids] @ q

    # ---------- Algorithm 2: best-first search on one layer ----------
    def search_layer(self, q, entries, ef, layer):
        visited = set(entries)
        d = self._dist(q, entries)
        cand = list(zip(d.tolist(), entries)); heapq.heapify(cand)            # min-heap
        best = [(-x, e) for x, e in cand]; heapq.heapify(best)                # max-heap (negated)
        while cand:
            dc, c = heapq.heappop(cand)
            if dc > -best[0][0]:
                break
            new = [n for n in self.neighbors(c, layer) if n not in visited]
            if not new:
                continue
            visited.update(new)
            for dn, n in zip(self._dist(q, new).tolist(), new):
                if len(best) < ef or dn < -best[0][0]:
                    heapq.heappush(cand, (dn, n))
                    if len(best) < ef:
                        heapq.heappush(best, (-dn, n))
                    else:
                        heapq.heappushpop(best, (-dn, n))                     # push + drop worst
        return heapq.nlargest(len(best), best)                                # [(−dist, id)] closest first

    # ---------- Algorithm 4: neighbor selection heuristic ----------
    def select_neighbors(self, ids, dists, m):
        """ids sorted by dists (ascending). Keep c only if it's closer to the base
        than to every already-kept neighbor."""
        kept = []
        for c, d in zip(ids, dists):
            if len(kept) == m:
                break
            if not kept or d < (1 - self.vecs[kept] @ self.vecs[c]).min():  # vectorized check
                kept.append(int(c))
        return kept

    # ---------- Algorithm 1: insert ----------
    def insert(self, vec):
        if self.n == len(self.vecs):
            self._grow()
        q = np.array(vec, dtype=np.float32)                           # copy: don't mutate caller data
        q /= np.linalg.norm(q)
        nid = self.n; self.n += 1
        self.vecs[nid] = q

        level = int(self.rng.geometric(1 - 1 / self.M)) - 1           # P(level ≥ L) = (1/M)^L
        while len(self.upper) < level:
            self.upper.append({})
        for l in range(1, level + 1):
            self.upper[l - 1][nid] = []

        if self.entry == -1:                                          # first node
            self.entry, self.max_level = nid, level
            return nid

        ep = [self.entry]                                             # greedy descent
        for l in range(self.max_level, level, -1):
            ep = [self.search_layer(q, ep, 1, l)[0][1]]

        for l in range(min(level, self.max_level), -1, -1):           # search, select, link
            res = self.search_layer(q, ep, self.efc, l)
            ids = [c for _, c in res]
            dists = [-x for x, _ in res]
            nbrs = self.select_neighbors(ids, dists, self.M)
            self.set_neighbors(nid, l, nbrs)

            cap = self.M0 if l == 0 else self.M
            for n in nbrs:                                            # reverse links
                cur = np.append(self.neighbors(n, l), nid).astype(np.int32)
                if len(cur) > cap:                                    # overflow → re-prune
                    dn = self._dist(self.vecs[n], cur)
                    order = np.argsort(dn)
                    cur = self.select_neighbors(cur[order], dn[order], cap)
                self.set_neighbors(n, l, cur)
            ep = ids

        if level > self.max_level:
            self.entry, self.max_level = nid, level
        return nid

    # ---------- Algorithm 5: k-NN search ----------
    def search(self, vec, k=5, ef_search=50):
        q = np.array(vec, dtype=np.float32)
        q /= np.linalg.norm(q)
        ep = [self.entry]
        for l in range(self.max_level, 0, -1):
            ep = [self.search_layer(q, ep, 1, l)[0][1]]
        return [(-x, int(c)) for x, c in self.search_layer(q, ep, max(ef_search, k), 0)[:k]]


def brute_force(X_normed, q, k):
    """Exact top-k by cosine, via argpartition (O(n)) then sort just those k."""
    s = X_normed @ (q / np.linalg.norm(q))
    top = np.argpartition(-s, k)[:k]
    return top[np.argsort(-s[top])]


    


In [20]:
import time
rng = np.random.default_rng(0)

idx = HNSW(M.shape[1], M=16, ef_construction=100, capacity=256, seed=0)
t = time.time()
for emb in M:
	idx.insert(emb)
print(f"built {idx.n} in {time.time()-t:.1f}s, top layer={idx.max_level}, "
		f"nodes per upper layer={[len(u) for u in idx.upper]}")

Xn = M / np.linalg.norm(M, axis=1, keepdims=True)
for ef in (50, 200):
	hits = sum(len(set(brute_force(Xn, q, 10)) & {i for _, i in idx.search(q, 10, ef)}) for q in M)
	print(f"recall@10 (ef_search={ef}) = {hits/1000:.1%}")

built 20 in 0.0s, top layer=0, nodes per upper layer=[]
recall@10 (ef_search=50) = 20.0%
recall@10 (ef_search=200) = 20.0%


In [21]:
q = model.encode("algebraic topology", normalize_embeddings=True).astype(np.float32)
idx.search(q, 10, ef)

[(0.8964588642120361, 0),
 (0.905188798904419, 19),
 (0.9093209505081177, 14),
 (0.9134501218795776, 10),
 (0.9313461184501648, 3),
 (0.9331603050231934, 5),
 (0.9371911287307739, 8),
 (0.9601510167121887, 12),
 (0.9667799472808838, 1),
 (0.974983811378479, 17)]